# 02 - Data Cleaning
**Vehicle Predictive Maintenance Project**

---

## Purpose of This Notebook

In notebook 01 we explored all five datasets and identified a clear set of issues to resolve before any modelling can take place. This notebook works through each of those issues systematically, documenting every decision made along the way.

The goal is to produce clean, joined dataset saved to `data/processed/` that notebook 03 (Feature Engineering) can pick up and use directly.

**Decisions carried forward from EDA:**
- Filter to vans only (Medium Van, Small Van, Large Van)
- Exclude 2026 maintenance records — submission lag makes them unreliable
- Exclude Environmental Disposal records — not mechanical repairs
- Driver Scores: filter to May 2023 onwards — pre-2023-05 scores were a system default of 100
- Branch: use Driver_Scores as the source of truth — Vehicle_Info branch names are outdated
- Missing driver assignments: fill with fleet average driver score
- Missing branch on Vehicle_Driver: fill from Driver_Scores branch lookup

## 1. Setup & Imports

We load the same libraries as notebook 01. Nothing new here — we are just reading raw files and writing cleaned ones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

print('Setup complete.')

## 2. Load Raw Data

We load all five files exactly as they are. No changes at this stage — we want to preserve the raw data and apply all transformations in a traceable way.

In [ ]:
vehicle_info    = pd.read_excel(RAW + 'Vehicle_Info.xlsx')
vehicle_driver  = pd.read_excel(RAW + 'Vehicle_Driver.xlsx')
vehicle_mileage = pd.read_excel(RAW + 'Vehicle_Mileage.xlsx')
maintenance     = pd.read_excel(RAW + 'Maintenance_Records.xlsx')
driver_scores   = pd.read_excel(RAW + 'Driver_Scores.xlsx')

print('All raw files loaded.')
for name, df in [('Vehicle_Info', vehicle_info), ('Vehicle_Driver', vehicle_driver),
                 ('Vehicle_Mileage', vehicle_mileage), ('Maintenance_Records', maintenance),
                 ('Driver_Scores', driver_scores)]:
    print(f'  {name:25s} {df.shape[0]:>6,} rows x {df.shape[1]} cols')

## 3. Clean Vehicle Info

### Why
Vehicle_Info is our master vehicle reference table. It contains every vehicle the business has ever operated. Before we do anything else we need to:
1. Filter down to vans only — cars and motorcycles are out of scope for this project
2. Derive vehicle age, which will be a key feature in the models
3. Standardise the registration number column name so it matches across all datasets

### Decision: Disposed Vehicles
We keep disposed vehicles in this cleaned table. They have a full repair history which is valuable training data — the model can learn from vehicles that have already gone through their full lifecycle. When we get to the prediction output stage we will filter to active vehicles only.

In [ ]:
# Filter to vans only
van_types = ['Medium Van', 'Small Van', 'Large Van']
vi = vehicle_info[vehicle_info['Asset Type'].isin(van_types)].copy()

print(f'Before filter: {len(vehicle_info):,} vehicles')
print(f'After van filter: {len(vi):,} vehicles')
print(f'Removed: {len(vehicle_info) - len(vi):,} non-van records\n')
print('Asset Type breakdown after filter:')
print(vi['Asset Type'].value_counts())

In [ ]:
# Convert registered date and derive vehicle age
vi['Registered Date'] = pd.to_datetime(vi['Registered Date'], errors='coerce')
vi['Vehicle Age (Years)'] = (pd.Timestamp.now() - vi['Registered Date']).dt.days / 365.25
vi['Vehicle Age (Years)'] = vi['Vehicle Age (Years)'].round(2)

# Standardise column name for joining
vi = vi.rename(columns={'RegistrationNumber': 'Registration'})

print('Vehicle age summary:')
print(vi['Vehicle Age (Years)'].describe())
print(f'\nMissing Registered Date: {vi["Registered Date"].isnull().sum()}')

In [ ]:
# Active vs Disposed split — for reference
print('Vehicle status breakdown (vans only):')
print(vi['Vehicle Status'].value_counts())
print('\n>> Both Active and Disposed are kept. Disposed vehicles provide training data.')
print('>> Active filter will be applied in the prediction output notebook only.')

## 4. Clean Driver Scores

### Why
Driver Scores is a weekly table with one row per driver per week. Before we can join this to vehicles we need to aggregate it to a single score per driver.

### Decision: Filter to May 2023 Onwards
EDA showed that scores before May 2023 were a system default of 100 — not real measured behaviour. Including this data would artificially inflate scores for long-tenured drivers and make the averages meaningless. We cut the data at 2023-05-01.

### Decision: Branch Source
We use the Branch Name from Driver_Scores as our branch reference rather than Vehicle_Info. The Driver_Scores table is actively maintained and reflects current branch structures. Vehicle_Info branch names are historic and may reference old branch names that no longer exist.

In [ ]:
# Convert week to datetime and filter to May 2023 onwards
driver_scores['Week'] = pd.to_datetime(driver_scores['Week'], errors='coerce')

cutoff = pd.Timestamp('2023-05-01')
print(f'Records before {cutoff.date()}: {(driver_scores["Week"] < cutoff).sum():,}')

ds = driver_scores[driver_scores['Week'] >= cutoff].copy()
print(f'Records after filter: {len(ds):,}')
print(f'Date range: {ds["Week"].min().date()} to {ds["Week"].max().date()}')

In [ ]:
# Convert sub-score columns to numeric (stored as strings in raw data)
count_cols = ['DeccelerationCount', 'AccelerationCount', 'SpeedingCount', 'CorneringCount']
rate_cols  = ['DeccelerationRate', 'AccelerationRate', 'SpeedingRate', 'CorneringRate']

for col in count_cols + rate_cols:
    ds[col] = pd.to_numeric(ds[col], errors='coerce')

print('Sub-score columns converted to numeric.')
print('Missing values after conversion:')
print(ds[count_cols + rate_cols].isnull().sum())

In [ ]:
# Aggregate to one row per driver
# We take the mean across all weeks for each score metric
# Branch is taken as the most recent branch recorded for that driver
driver_agg = ds.groupby('DriverName').agg(
    Branch           = ('Branch Name', lambda x: x.dropna().iloc[-1] if len(x.dropna()) > 0 else np.nan),
    DriverScore      = ('DriverScore', 'mean'),
    DeccelerationRate= ('DeccelerationRate', 'mean'),
    AccelerationRate = ('AccelerationRate', 'mean'),
    SpeedingRate     = ('SpeedingRate', 'mean'),
    CorneringRate    = ('CorneringRate', 'mean'),
    WeeksRecorded    = ('Week', 'count')
).reset_index()

driver_agg['DriverScore'] = driver_agg['DriverScore'].round(1)

fleet_avg_score = driver_agg['DriverScore'].mean().round(1)

print(f'Unique drivers after aggregation: {len(driver_agg):,}')
print(f'Fleet average driver score: {fleet_avg_score}')
print(f'\nDriver score summary:')
print(driver_agg['DriverScore'].describe())

## 5. Clean Vehicle Driver Table

### Why
Vehicle_Driver gives us the current driver-to-vehicle assignment and the next MOT date. There are two data quality issues to resolve:

1. **Missing Driver (92 records)** — some vehicles have no driver assigned. Rather than dropping these vehicles, we will assign them the fleet average driver score. This is a conscious simplification — it means unassigned vehicles are treated as average-risk from a driving behaviour perspective.

2. **Missing Branch (57 records)** — we will fill these from the Driver_Scores branch lookup using the driver name as the key. If a driver has no branch in Driver_Scores either, the branch will remain null and be handled in the feature engineering stage.

### Decision: Van Filter
We apply the same van-only filter here. Vehicle_Driver also contains cars and motorcycles which we don't need.

In [ ]:
# Filter to vans only and standardise registration column
vd = vehicle_driver[vehicle_driver['Asset Type'].isin(van_types)].copy()
vd = vd.rename(columns={'RegistrationNumber': 'Registration'})

print(f'Vehicle_Driver after van filter: {len(vd):,} records')
print(f'Missing Driver: {vd["Driver"].isnull().sum()}')
print(f'Missing Branch: {vd["Branch"].isnull().sum()}')

In [ ]:
# Fill missing Branch from Driver_Scores branch lookup
driver_branch_lookup = driver_agg[['DriverName', 'Branch']].rename(columns={'DriverName': 'Driver'})

vd = vd.merge(driver_branch_lookup, on='Driver', how='left', suffixes=('', '_from_scores'))

# Where Branch is missing, fill from Driver_Scores lookup
vd['Branch'] = vd['Branch'].fillna(vd['Branch_from_scores'])
vd = vd.drop(columns=['Branch_from_scores'])

print(f'Missing Branch after fill: {vd["Branch"].isnull().sum()}')
print('>> Any remaining nulls are vehicles with no driver and no branch in either source.')

In [ ]:
# Clean MOT date
vd['Next MOT Date'] = pd.to_datetime(vd['Next MOT Date'], errors='coerce')
today = pd.Timestamp.now().normalize()
vd['Days Until MOT'] = (vd['Next MOT Date'] - today).dt.days

print('MOT date cleaning complete.')
print(f'Missing MOT dates: {vd["Next MOT Date"].isnull().sum()}')
print(f'MOT overdue: {(vd["Days Until MOT"] < 0).sum()}')
print(f'MOT due within 90 days: {vd["Days Until MOT"].between(0, 90).sum()}')

In [ ]:
# Keep only the columns we need from Vehicle_Driver
vd_clean = vd[['Registration', 'Driver', 'Branch', 'Next MOT Date', 'Days Until MOT']].copy()

print('Vehicle_Driver cleaned columns:')
print(vd_clean.dtypes)
display(vd_clean.head(5))

## 6. Clean Vehicle Mileage

### Why
Vehicle_Mileage records monthly mileage per vehicle. In EDA we found that `Average Miles Per Day` and `Year` are stored as object (string) types rather than numeric — this suggests there are some non-numeric values mixed in that need to be handled.

We need to produce two outputs from this table:
1. A **cumulative mileage** figure per vehicle — total miles driven across all recorded months, used as a wear indicator in the model
2. A **monthly mileage lookup** — used later when we want to estimate mileage at the time of a specific repair

In [ ]:
vm = vehicle_mileage.copy()
vm = vm.rename(columns={'RegistrationNumber': 'Registration'})

# Convert to numeric — any values that can't be converted become NaN
vm['Average Miles Per Day'] = pd.to_numeric(vm['Average Miles Per Day'], errors='coerce')
vm['Year'] = pd.to_numeric(vm['Year'], errors='coerce')
vm['Total Miles'] = pd.to_numeric(vm['Total Miles'], errors='coerce')

print('After numeric conversion:')
print(f'Missing Average Miles Per Day: {vm["Average Miles Per Day"].isnull().sum()}')
print(f'Missing Year: {vm["Year"].isnull().sum()}')
print(f'Missing Total Miles: {vm["Total Miles"].isnull().sum()}')

In [ ]:
# Create a proper date column from Year + Month for time-based operations
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

vm['Month_Num'] = vm['Month'].map(month_map)
vm['Period'] = pd.to_datetime(dict(year=vm['Year'], month=vm['Month_Num'], day=1), errors='coerce')

missing_period = vm['Period'].isnull().sum()
print(f'Missing Period dates after construction: {missing_period}')
if missing_period > 0:
    print('Dropping rows with unparseable dates...')
    vm = vm.dropna(subset=['Period'])

print(f'\nMileage date range: {vm["Period"].min().date()} to {vm["Period"].max().date()}')

In [ ]:
# Filter to vans only
van_regs = set(vi['Registration'])
vm = vm[vm['Registration'].isin(van_regs)].copy()
print(f'Mileage records after van filter: {len(vm):,}')

# Cumulative mileage per vehicle — sum of all Total Miles recorded
cumulative_mileage = vm.groupby('Registration')['Total Miles'].sum().reset_index()
cumulative_mileage.columns = ['Registration', 'Cumulative Miles']

print(f'\nVehicles with mileage data: {len(cumulative_mileage):,}')
print('\nCumulative Miles summary:')
print(cumulative_mileage['Cumulative Miles'].describe())

## 7. Clean Maintenance Records

### Why
Maintenance_Records is the heart of this project — it's the source of truth for what has gone wrong with each vehicle and when. Several cleaning steps are needed:

1. **Filter to vans only** — match to our van registration list
2. **Exclude 2026 records** — as identified in EDA, 2026 records are incomplete due to submission lag and would create a misleading drop in recent activity
3. **Exclude Environmental Disposal** — these are waste disposal charges, not mechanical repairs, and would distort repair frequency and cost analysis
4. **Convert date columns** — Invoice Date is the most reliable date field and will be our primary date reference
5. **Handle negative/zero costs** — flag these for review; they may be credits or data entry errors

In [ ]:
mr = maintenance.copy()
mr = mr.rename(columns={'Registration Number': 'Registration'})

# Convert date columns
mr['Invoice Date'] = pd.to_datetime(mr['Invoice Date'], errors='coerce')
mr['Booking Date'] = pd.to_datetime(mr['Booking Date'], errors='coerce')
mr['Incident Start Date'] = pd.to_datetime(mr['Incident Start Date'], errors='coerce')

print(f'Raw maintenance records: {len(mr):,}')
print(f'Invoice Date range: {mr["Invoice Date"].min().date()} to {mr["Invoice Date"].max().date()}')

In [ ]:
# Filter to vans only
mr = mr[mr['Registration'].isin(van_regs)].copy()
print(f'After van filter: {len(mr):,} records')

In [ ]:
# Exclude 2026 records — incomplete submission data
before_2026 = len(mr)
mr = mr[mr['Invoice Date'].dt.year < 2026].copy()
print(f'Records removed for 2026 (submission lag): {before_2026 - len(mr):,}')
print(f'Remaining: {len(mr):,}')

In [ ]:
# Exclude Environmental Disposal — not mechanical repairs
env_mask = mr['category'].str.contains('Environmental Disposal', na=False)
print(f'Environmental Disposal records removed: {env_mask.sum():,}')
mr = mr[~env_mask].copy()
print(f'Remaining: {len(mr):,}')

In [ ]:
# Review zero and negative Net Amount records
zero_or_neg = mr[mr['Net Amount'] <= 0]
print(f'Zero or negative Net Amount records: {len(zero_or_neg)}')
if len(zero_or_neg) > 0:
    print('\nCategory breakdown of zero/negative records:')
    print(zero_or_neg['Top Category'].value_counts())
    print('\n>> These may be warranty claims, credits, or data entry errors.')
    print('>> We keep them for repair frequency analysis but exclude from cost modelling.')

In [ ]:
# Final category breakdown after cleaning
print('Maintenance records by Top Category after cleaning:')
print(mr['Top Category'].value_counts())

In [ ]:
# Keep relevant columns only
mr_clean = mr[['Registration', 'Job Number', 'Invoice Date', 'Booking Date', 'Top Category', 'category', 'Item', 'Net Amount']].copy()

print('Maintenance Records cleaned.')
print(f'Final shape: {mr_clean.shape}')
display(mr_clean.head(5))

## 8. Build the Master Vehicle Table

### Why
Now that each individual dataset is clean, we join them together into a single master vehicle table. This will be the foundation for feature engineering in notebook 03.

The join strategy is:
- **Vehicle_Info** is the master — every van starts here
- **Vehicle_Driver** is joined to bring in current driver, branch, and MOT date — left join so we keep all vans even if they have no current driver record
- **Cumulative Mileage** is joined on registration — left join again
- **Driver Scores** are joined via the driver name — vehicles with no driver get the fleet average score

Maintenance records are kept as a separate clean table because they are one-to-many (a vehicle has many repairs). They will be aggregated into features in the next notebook.

In [ ]:
# Start with Vehicle_Info as master
master = vi[['Registration', 'Asset Type', 'Vehicle Status', 'Make', 'Model',
             'Derivative', 'Registered Date', 'Vehicle Age (Years)']].copy()

print(f'Master table starting shape: {master.shape}')

In [ ]:
# Join Vehicle_Driver
master = master.merge(vd_clean, on='Registration', how='left')
print(f'After joining Vehicle_Driver: {master.shape}')
print(f'Missing Driver after join: {master["Driver"].isnull().sum()}')

In [ ]:
# Join cumulative mileage
master = master.merge(cumulative_mileage, on='Registration', how='left')
print(f'After joining Mileage: {master.shape}')
print(f'Missing Cumulative Miles: {master["Cumulative Miles"].isnull().sum()}')
print('>> Vehicles with no mileage data are likely very recently acquired.')

In [ ]:
# Join Driver Scores via driver name
driver_score_lookup = driver_agg[['DriverName', 'DriverScore', 'DeccelerationRate',
                                   'AccelerationRate', 'SpeedingRate', 'CorneringRate']].copy()
driver_score_lookup = driver_score_lookup.rename(columns={'DriverName': 'Driver'})

master = master.merge(driver_score_lookup, on='Driver', how='left')
print(f'After joining Driver Scores: {master.shape}')
print(f'Missing DriverScore before fill: {master["DriverScore"].isnull().sum()}')

In [ ]:
# Fill missing driver scores with fleet average
# This applies to: vehicles with no driver assigned, and drivers not found in Driver_Scores
master['DriverScore'] = master['DriverScore'].fillna(fleet_avg_score)
master['DeccelerationRate'] = master['DeccelerationRate'].fillna(driver_agg['DeccelerationRate'].mean())
master['AccelerationRate']  = master['AccelerationRate'].fillna(driver_agg['AccelerationRate'].mean())
master['SpeedingRate']      = master['SpeedingRate'].fillna(driver_agg['SpeedingRate'].mean())
master['CorneringRate']     = master['CorneringRate'].fillna(driver_agg['CorneringRate'].mean())

print(f'Missing DriverScore after fill: {master["DriverScore"].isnull().sum()}')
print(f'Fleet average score applied: {fleet_avg_score}')

In [ ]:
active = master[master['Vehicle Status'] == 'Current - On Road']
print(f'Active vehicles: {len(active)}')
print('\nNull check on ACTIVE vehicles only:')
nulls = active.isnull().sum()
print(nulls[nulls > 0])


## 9. Save Cleaned Data

### Why
We save everything to `data/processed/` so that notebook 03 has clean, ready-to-use inputs. This separation is important — if we ever need to rerun just the feature engineering or modelling stages, we don't need to re-run all the cleaning steps.

We save:
- `master_vehicles.csv` — one row per van, all joined vehicle and driver info
- `maintenance_clean.csv` — cleaned repair records (one row per repair event)
- `mileage_clean.csv` — monthly mileage records for all vans
- `driver_scores_clean.csv` — aggregated driver scores for reference

In [ ]:
master.to_csv(PROCESSED + 'master_vehicles.csv', index=False)
mr_clean.to_csv(PROCESSED + 'maintenance_clean.csv', index=False)
vm.to_csv(PROCESSED + 'mileage_clean.csv', index=False)
driver_agg.to_csv(PROCESSED + 'driver_scores_clean.csv', index=False)

print('All cleaned files saved to data/processed/:')
print(f'  master_vehicles.csv     — {master.shape[0]:,} rows x {master.shape[1]} cols')
print(f'  maintenance_clean.csv   — {mr_clean.shape[0]:,} rows x {mr_clean.shape[1]} cols')
print(f'  mileage_clean.csv       — {vm.shape[0]:,} rows x {vm.shape[1]} cols')
print(f'  driver_scores_clean.csv — {driver_agg.shape[0]:,} rows x {driver_agg.shape[1]} cols')

## 10. Cleaning Summary

A final record of every decision made in this notebook for traceability.

In [ ]:
print("""
========================================================
 CLEANING SUMMARY
========================================================

VEHICLE INFO:
  - Filtered to Medium Van, Small Van, Large Van only
  - Derived Vehicle Age (Years) from Registered Date
  - Disposed vehicles retained for training data

DRIVER SCORES:
  - Filtered to Week >= 2023-05-01 (pre-date was constant 100)
  - Aggregated weekly rows to single average per driver
  - Branch taken from most recent Driver_Scores record per driver
  - Sub-score count columns converted from string to numeric

VEHICLE DRIVER:
  - Filtered to vans only
  - Missing Branch filled from Driver_Scores branch lookup
  - MOT date converted, Days Until MOT calculated

VEHICLE MILEAGE:
  - Average Miles Per Day and Year converted to numeric
  - Period date column constructed from Year + Month
  - Cumulative Miles calculated per vehicle

MAINTENANCE RECORDS:
  - Filtered to vans only
  - 2026 records excluded (submission lag — not yet complete)
  - December 2025 retained (genuine seasonal dip)
  - Environmental Disposal excluded (not mechanical repairs)
  - Zero/negative costs flagged (retained for frequency, excluded from cost modelling)

MASTER TABLE:
  - Vehicle_Info as base, joined with Driver, Mileage, Driver Scores
  - Missing driver scores filled with fleet average
  - Missing sub-scores filled with fleet averages
========================================================
""")